# 06 · Baseline retrieval: BM25

Task B: a new petition arrives; which open petition does it duplicate? For every validation duplicate, BM25 ranks
the petitions in its **time-aware pool** (04: published petitions open when the moderator decided) and we check
where the correct petition(s) land. A moderator would look at the top 5.

**Input** (in `data/processed/`): `petitions.parquet`, `gold_pairs.parquet`, `splits.parquet`.
The BM25 code, the ranking rule and the metrics are in `src/retrieval.py`, so 07 (sentence vectors), 09 and the
pipeline use exactly the same ones.

**Output:**

| File | Content |
|---|---|
| `data/processed/predictions/bm25_val_ranks.parquet` | per validation query: rank of the first correct petition in the open and the stress pool, pool sizes |
| `reports/bm25_val_results.csv` | Recall@1/5/10 and MRR for each k1 (open pool) and for the stress pool |

Steps: 1 load → 2 BM25 in plain words → 3 index → 4 validation queries → 5 choose k1 → 6 stress test →
7 where BM25 fails → 8 save. Runs on the laptop in about 2 minutes. The test split is not used.

In [1]:
import sys, time
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent                      # this notebook runs inside notebooks/
sys.path.insert(0, str(ROOT / "src"))
from preprocess import MODEL_INPUTS, leakage_check
from splits import CandidatePools
from retrieval import BM25Index, bm25_tokens, gold_rank, retrieval_metrics

DATA = ROOT / "data" / "processed"
K1_VALUES = [1.2, 1.5, 2.0]    # locked range "k1 about 1.2-2.0"; the best MRR on validation wins
B = 0.75                       # locked
SEED = 42

## 1. Load

In [2]:
columns = ["id", "label", "action", "text", "decision_time", "opened_at", "closed_at"]
petitions = pd.read_parquet(DATA / "petitions.parquet", columns=columns)
petitions = petitions.merge(pd.read_parquet(DATA / "splits.parquet"), on="id", validate="one_to_one")
gold = pd.read_parquet(DATA / "gold_pairs.parquet")
assert (len(petitions), len(gold)) == (127_191, 61_132)

pools = CandidatePools(petitions)             # the pool rule of 04 (published, open at t, never the query itself)
by_id = petitions.set_index("id")
print(f"{len(petitions):,} petitions, {len(pools.ids):,} published (possible pool members), {len(gold):,} gold pairs")

127,191 petitions, 40,381 published (possible pool members), 61,132 gold pairs


## 2. BM25 in plain words

**Text → stems.** Lower-case, keep words of 2+ characters, drop stop words ("the", "should", "more", …: the 318
words of scikit-learn's English list), then cut each word to its stem with the Porter stemmer, so "licensing" and
"licensed" both become "licens".

In [3]:
for pid in [150557, 121973]:
    print(f"#{pid}  {by_id.loc[pid, 'action']!r:32s} -> {bm25_tokens(by_id.loc[pid, 'action'])}")

#150557  'Ban the t.v licence'            -> ['ban', 'licenc']
#121973  'Stop TV licensing!'             -> ['stop', 'tv', 'licens']


This gold pair from the data audit ("Ban the t.v licence" was rejected as a duplicate of "Stop TV licensing!")
shares **no stem** in the titles: "t.v" splits into single letters, and "licence" / "licensing" stem differently.
Word matching cannot connect such paraphrases; this is why 07 tests sentence vectors.

**Score.** For a query, every petition in the pool gets a score that adds up, over the query's stems:

- **idf**: a stem that is rare in the pool counts more than a common one: ln(1 + (N − df + 0.5) / (df + 0.5)),
  with N petitions in the pool and df of them containing the stem. The "1 +" keeps it positive.
- **term frequency with saturation (k1)**: the first occurrence in a petition counts most, each further one adds
  less. A small k1 saturates fast (almost "present or not"), a large k1 lets repeats count longer.
- **length normalisation (b = 0.75)**: the same match counts a little less in a long petition than in a short one.

N, df and the average length are computed **inside each query's pool**, so petitions outside the pool, including
later ones, never influence the score.

**Metrics.** For each query we find the rank of the first correct petition (several correct petitions: the
best-ranked one counts; ties count against BM25). **Recall@k** = share of queries with a correct petition in the
top k; **MRR** = average of 1 / rank. A query whose correct petitions are all outside its pool counts as a miss.

## 3. Index of the published petitions

In [4]:
published_text = by_id.loc[pools.ids, "text"]     # rows in the same order as every pool mask
leakage_check(published_text)                       # only the text is used
start = time.time()
index = BM25Index(published_text.tolist())
print(f"{index.counts.shape[0]:,} petitions x {index.counts.shape[1]:,} stems, "
      f"median {np.median(index.doc_len):.0f} stems per petition; built in {time.time() - start:.0f} s")

40,381 petitions x 27,861 stems, median 59 stems per petition; built in 3 s


## 4. Validation queries

The validation duplicates with at least one valid link (02), with all their correct petitions.

In [5]:
val_queries = petitions[(petitions["split"] == "val") & petitions["id"].isin(gold["query_id"])]
gold_sets = gold.groupby("query_id")["target_id"].apply(set)
leakage_check(val_queries[MODEL_INPUTS])
assert len(val_queries) == 1_155

open_sizes = pools.sizes(val_queries["decision_time"], val_queries["id"])
stress_sizes = pools.sizes(val_queries["decision_time"], val_queries["id"], stress=True)
several = (val_queries["id"].map(gold_sets).map(len) > 1).sum()
print(f"{len(val_queries):,} queries ({several} with several correct petitions); "
      f"median pool: open {np.median(open_sizes):,.0f}, stress {np.median(stress_sizes):,.0f}")

1,155 queries (102 with several correct petitions); median pool: open 1,897, stress 36,241


## 5. Choose k1 on the open pool

Each k1 of the locked range is tried with b = 0.75; the best **MRR** wins. MRR is used for the choice because it
looks at the whole ranking (Recall@k only asks "in the top k or not").

In [6]:
def evaluate(queries, k1, stress=False):
    """Rank of the first correct petition for every query, and milliseconds per query."""
    ranks, start = [], time.time()
    for query_id, text, t in zip(queries["id"], queries["text"], queries["decision_time"]):
        mask = pools.mask(t, query_id, stress)
        scores = index.scores(text, mask, k1=k1, b=B)
        ranks.append(gold_rank(scores, pools.ids[mask], gold_sets[query_id]))
    return np.array(ranks, dtype=float), (time.time() - start) / len(queries) * 1000

rows, ranks_by_k1 = [], {}
for k1 in K1_VALUES:
    ranks, ms = evaluate(val_queries, k1)
    ranks_by_k1[k1] = ranks
    rows.append({"pool": "open", "k1": k1, **retrieval_metrics(ranks), "ms_per_query": round(ms, 1)})
results = pd.DataFrame(rows)
BEST_K1 = float(results.loc[results["mrr"].idxmax(), "k1"])
print("best k1:", BEST_K1)
results.round(4)

best k1: 2.0


,pool,k1,recall@1,recall@5,recall@10,mrr,ms_per_query
0,open,1.2,0.5065,0.7879,0.8597,0.6310,3.9
1,open,1.5,0.5126,0.7957,0.8615,0.6388,3.9
2,open,2.0,0.5238,0.8043,0.8719,0.6488,3.9


## 6. Stress test: the all-published pool

The same queries against **every** published petition opened by t + 48 h, also the ones already closed
(about 20× larger pools). This shows how much the time-aware pool helps, and how BM25 copes with many more
look-alike petitions.

In [7]:
stress_ranks, ms = evaluate(val_queries, BEST_K1, stress=True)
rows.append({"pool": "stress", "k1": BEST_K1, **retrieval_metrics(stress_ranks), "ms_per_query": round(ms, 1)})
results = pd.DataFrame(rows)
best = results[results["k1"] == BEST_K1].set_index("pool").drop(columns="k1")
best.round(4)

,recall@1,recall@5,recall@10,mrr,ms_per_query
pool,,,,,
open,0.5238,0.8043,0.8719,0.6488,3.9
stress,0.1680,0.3905,0.5177,0.2813,25.4


## 7. Where BM25 fails

Where the first correct petition lands, and a few validation misses side by side.

In [8]:
open_ranks = ranks_by_k1[BEST_K1]
bins = [0, 1, 5, 10, 50, np.inf]
labels = ["1", "2-5", "6-10", "11-50", "> 50"]
landing = pd.cut(open_ranks[np.isfinite(open_ranks)], bins=bins, labels=labels).value_counts().reindex(labels)
landing["not in the pool"] = int(np.isinf(open_ranks).sum())
pd.DataFrame({"queries": landing, "share": (landing / len(open_ranks)).round(3)})

,queries,share
1,605,0.524
2-5,324,0.281
6-10,78,0.068
11-50,105,0.091
> 50,41,0.035
not in the pool,2,0.002


In [9]:
# The audit's paraphrase pair (a train query): where does BM25 put the correct petition?
query = by_id.loc[150557]
mask = pools.mask(query["decision_time"], 150557)
scores = index.scores(query["text"], mask, k1=BEST_K1, b=B)
pool_ids = pools.ids[mask]
print(f"#150557 {query['action']!r}: correct petition #121973 at rank "
      f"{gold_rank(scores, pool_ids, gold_sets[150557]):.0f} of {mask.sum():,}; BM25's top 3:")
for pid in pool_ids[np.argsort(-scores)[:3]]:
    print(f"   #{pid} {by_id.loc[pid, 'action']!r}")

#150557 'Ban the t.v licence': correct petition #121973 at rank 173 of 3,187; BM25's top 3:
   #125635 'All illegal immigrants to be deported within 48 hours & a 10yr ban from a visa!'
   #124937 "Decriminalise None Payment Of The BBC's Television Licence (TV Licence)"
   #128843 'Ban diesel fuel! We deserve cleaner air.'


In [10]:
# Five random validation misses (correct petition below the top 10)
ranked = val_queries.assign(rank=open_ranks)
missed = ranked[ranked["rank"] > 10].sample(5, random_state=SEED)
pd.DataFrame({"query": missed["action"].to_numpy(),
              "correct petition": [by_id.loc[min(gold_sets[q]), "action"] for q in missed["id"]],
              "rank": missed["rank"].to_numpy()})

,query,correct petition,rank
0,Formally approach the United Nations to deploy...,Urge UN to send peacekeepers into Gaza & the W...,13.0
1,Government to hold a referendum on Digital ID,Hold a referendum on introducing mandatory ide...,37.0
2,Stop all arms exports to Israel immediately an...,Stop all armaments exports to Israel,20.0
3,We dont think that pay per mile has legitimate...,Do not introduce pay-per-mile road charging fo...,15.0
4,Request for the Dissolution of Parliament and ...,Call an immediate general election,93.0


## 8. Save

In [11]:
(DATA / "predictions").mkdir(exist_ok=True)
pd.DataFrame({"query_id": val_queries["id"].to_numpy(), "rank_open": open_ranks, "rank_stress": stress_ranks,
              "pool_open": open_sizes, "pool_stress": stress_sizes}).to_parquet(DATA / "predictions" / "bm25_val_ranks.parquet", index=False)
results.round(4).to_csv(ROOT / "reports" / "bm25_val_results.csv", index=False)
print("saved; best k1 =", BEST_K1)

saved; best k1 = 2.0


## Summary

- **Index:** 40,381 published petitions with 27,861 distinct stems (median 59 stems per petition), built in seconds.
- **Validation queries:** 1,155 duplicates, 102 of them with several correct petitions; median open pool 1,897.
- **k1:** MRR 0.631 (1.2), 0.639 (1.5) and 0.649 (2.0), so **k1 = 2.0** (b = 0.75). It is the upper end of the
  locked range; a larger k1 was not tried.
- **Open pool (main result):** Recall@1 **0.524**, Recall@5 **0.804**, Recall@10 **0.872**, MRR **0.649**, about
  4 ms per query. For 4 of 5 duplicates, the correct petition is among the 5 a moderator would see.
- **Stress pool** (all published petitions, median 36,241): Recall@5 falls to 0.391 and MRR to 0.281. Many older,
  closed petitions on the same topics compete, so the time-aware pool is essential.
- **Misses are mostly paraphrases:** the same request in other words ("Dissolution of Parliament" → "Call an
  immediate general election", rank 93; "Digital ID" → "mandatory identity", rank 37; "Ban the t.v licence" →
  "Stop TV licensing!", rank 173). BM25 only matches shared stems, which is why 07 tests sentence vectors.
  In some misses, other petitions on the same topic rank higher; some of them may also be valid duplicates that the
  moderator did not link, so the scores may understate how useful the list is.
- 2 queries (0.2%) have no correct petition in their open pool (the ceiling from 04: 99.83%).

Next: 07 runs the same evaluation with the MiniLM and mpnet vectors and compares them with BM25.
